In [1]:
!pip install torch

  Using cached setuptools-81.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached markupsafe-3.0.3-cp314-cp314-macosx_11_0_arm64.whl.metadata (2.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 23.7 MB/s  0:00:03 eta 0:00:010:01:01
Using cached setuptools-81.0.0-py3-none-any.whl (1.1 MB)
Using cached fsspec-2026.4.0-py3-none-any.whl (203 kB)
Using cached networkx-3.6.1-py3-none-any.whl (2.1 MB)
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
Using cached markupsafe-3.0.3-cp314-cp314-macosx_11_0_arm64.whl (12 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import torch

from torch.utils.data import Dataset, DataLoader


class GPTDatasetv1(Dataset):
    def __init__(self, txt, tokenizer, stride, max_length=512):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(txt)

        for i in range(0, len(token_ids) - max_length, stride):
            self.input_ids.append(token_ids[i : i + max_length])
            self.target_ids.append(token_ids[i + 1 : i + max_length + 1])

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        x = self.input_ids[idx]
        y = self.target_ids[idx]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

/Users/zero101010/Docs/build-llm-from-scratch/.venv/lib/python3.14/site-packages/torch/_subclasses/functional_tensor.py:362: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [3]:

def create_dataloader_v1(txt,batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    tiktokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetv1(txt, tiktokenizer, stride, max_length)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)
    return dataloader

In [ ]:
import requests

url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"

raw_text = requests.get(url).text

dataloader = create_dataloader_v1(raw_text,batch_size=1, max_length=3, stride=1, shuffle=False, drop_last=True)

data_iter = iter(dataloader)

first_batch = next(data_iter)
second_batch = next(data_iter)
print("First batch:\n", first_batch)

print(second_batch)



[tensor([[  40,  367, 2885]]), tensor([[ 367, 2885, 1464]])]
[tensor([[ 367, 2885, 1464]]), tensor([[2885, 1464, 1807]])]


In [35]:
data_iter = iter(dataloader)

for element in data_iter:
    inputs , targets = element
    print("Inputs:\n", inputs)  
    print("Targets:\n", targets)

Inputs:
 tensor([[  40,  367, 2885]])
Targets:
 tensor([[ 367, 2885, 1464]])
Inputs:
 tensor([[ 367, 2885, 1464]])
Targets:
 tensor([[2885, 1464, 1807]])
Inputs:
 tensor([[2885, 1464, 1807]])
Targets:
 tensor([[1464, 1807, 3619]])
Inputs:
 tensor([[1464, 1807, 3619]])
Targets:
 tensor([[1807, 3619,  402]])
Inputs:
 tensor([[1807, 3619,  402]])
Targets:
 tensor([[3619,  402,  271]])
Inputs:
 tensor([[3619,  402,  271]])
Targets:
 tensor([[  402,   271, 10899]])
Inputs:
 tensor([[  402,   271, 10899]])
Targets:
 tensor([[  271, 10899,  2138]])
Inputs:
 tensor([[  271, 10899,  2138]])
Targets:
 tensor([[10899,  2138,   257]])
Inputs:
 tensor([[10899,  2138,   257]])
Targets:
 tensor([[2138,  257, 7026]])
Inputs:
 tensor([[2138,  257, 7026]])
Targets:
 tensor([[  257,  7026, 15632]])
Inputs:
 tensor([[  257,  7026, 15632]])
Targets:
 tensor([[ 7026, 15632,   438]])
Inputs:
 tensor([[ 7026, 15632,   438]])
Targets:
 tensor([[15632,   438,  2016]])
Inputs:
 tensor([[15632,   438,  2016]])
Ta

In [29]:
dataloader = create_dataloader_v1(raw_text,batch_size=8, max_length=4, stride=4, shuffle=False, drop_last=True)

inputs , targets = next(iter(dataloader))
print("Inputs:\n", inputs)
print("Targets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])
